# Homework #4

By: Jenny Park\
Date: April 13, 2026\
Class: QMSS GR5069

In [0]:
import pandas as pd
from pyspark.sql import functions as F

In [0]:
!pip install xgboost

## **Prediction Problem:** Whether or not the driver will finish the race

#### Building the dataset

In [0]:
# loading in datasets
df_results = spark.read.csv('/Volumes/gr5069/raw/f1_data/results.csv', header=True)

df_drivers = spark.read.csv('/Volumes/gr5069/raw/f1_data/drivers.csv', header=True)

df_races = spark.read.csv('/Volumes/gr5069/raw/f1_data/races.csv', header=True)

df_constructors = spark.read.csv('/Volumes/gr5069/raw/f1_data/constructors.csv', header=True)

df_status = spark.read.csv('/Volumes/gr5069/raw/f1_data/status.csv', header=True)

In [0]:
# joining datasets
df = df_results.join(df_drivers, on='driverId', how='inner')
df = df.join(df_races, on='raceId', how='inner')
df = df.join(df_constructors, on='constructorId', how='inner')
df = df.join(df_status, on='statusId', how='inner')

# selecting features
df = df.select('constructorId', 'raceId', 'year', 'round', 'circuitId', 'driverId', 'grid', 'laps', 'statusId')
display(df)
df.columns

**Preprocessing steps**

In [0]:
# identify categorical columns
df = df.toPandas()

display(df)

categorical_columns = ['raceId', 'driverId', 'constructorId', 'circuitId', 'grid', 'year', 'round']

for col in categorical_columns:
    df[col] = df[col].astype('category')

numerical_columns = ['laps']

for col in numerical_columns:
    df[col] = df[col].astype('float')

df['statusId'] = df['statusId'].astype('int')
df.loc[df['statusId'] != 1, 'statusId'] = 0

df = df.select_dtypes(include=['category', 'float', 'int'])

In [0]:
display(df)
print(df['statusId'].value_counts())

**Perform a train/test split.**

In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop(['statusId'], axis=1), df[['statusId']].values.ravel(), random_state=42)

In [0]:
import mlflow.sklearn

from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
import seaborn as sns
import tempfile
import matplotlib.pyplot as plt


def log_xgb(experimentID, run_name, params, X_train, X_test, y_train, y_test):

    with mlflow.start_run(experiment_id=experimentID, run_name=run_name) as run:

        # Create model, train it, and create predictions
        model = XGBClassifier(**params)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        
        # Log model
        mlflow.sklearn.log_model(model, 'xgboost-model')
        
        # Log parameters
        [mlflow.log_param(param, value) for param, value in params.items()]

        # Create metrics
        accuracy = accuracy_score(y_test, predictions)
        f1 = f1_score(y_test, predictions)
        precision = precision_score(y_test, predictions)
        recall = recall_score(y_test, predictions)

        # Print metrics
        print(f'Accuracy: {accuracy_score(y_test, predictions)}')
        print(f'F1: {f1_score(y_test, predictions)}')
        print(f'Precision: {precision_score(y_test, predictions)}')
        print(f'Recall: {recall_score(y_test, predictions)}')

        # Log metrics
        mlflow.log_metric('accuracy', accuracy)
        mlflow.log_metric('f1', f1)
        mlflow.log_metric('precision', precision)
        mlflow.log_metric('recall', recall)

        # Create feature importance
        importance = pd.DataFrame(list(zip(X_train.columns, model.feature_importances_)), columns=['Feature', 'Importance']).sort_values('Importance', ascending=False)

        # Log importances using a temporary file
        temp = tempfile.NamedTemporaryFile(prefix='feature-importance-', suffix='.csv') 
        temp_name = temp.name
        try:
            importance.to_csv(temp_name, index=False)
            mlflow.log_artifact(temp_name, 'feature-importance.csv')
        finally:
            temp.close() # Delete the temp file

        # Create confusion matrix
        cm = confusion_matrix(y_test, predictions)
        fig, ax = plt.subplots()
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.xlabel('Predicted values for DNF')
        plt.ylabel('Actual values for DNF')
        plt.title('Confusion Matrix')

        # Log confusion matrices using a temporary file
        temp = tempfile.NamedTemporaryFile(prefix='confusion_matrix-', suffix='.png')
        temp_name = temp.name
        try:
            fig.savefig(temp_name)
            mlflow.log_artifact(temp_name, 'confusion_matrix.png')
        finally:
            temp.close() # Delete the temp file
        
        display(fig)
        return run.info.run_id

In [0]:
# default params
params = {
        'max_depth': 6,
        'learning_rate': 0.5,
        'gamma': 0
        'min_child_weight': 1,
        'max_delta_step': 0,
        'subsample': 1,
        'colsample_bytree': 1,
        'enable_categorical': True
}    

experiment = mlflow.set_experiment('/Users/jjp2219@columbia.edu/F1_XGBoost')
experiment_ID = experiment.experiment_id

log_xgb(experiment_ID, 'F1 XGBoost Parameter Tuning: Run 1', params, X_train, X_test, y_train, y_test)